[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/6_optimization.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

# Optimization
We will now see how we can optimize our system to handle different users.

During our lectures we've seen multiple types of user:
- coherent vs. incoherent
- under-informative vs. over-informative


## Questions
1. Which users can our system handle?
2. How can handle the remaining users?

### Solution

1. Let's consider our users one by one:

   - *Coherent*, the sytem should not do anything particular

   - *Incoherent*, we expect the NLU to extract new slots and the DST to update the slots with the new values

   - *Under-Informative*, depends on the number of slots provided. If the intent is clear, the system will ask for the missing slots. Otherwise, fall-back policy

   - *Over-Informative*, the system should be able to extract all slots at once... but, what if the user sentence has multiple intents/requests?

Consider the following: *"I would like to order two medium pizza since we are quite hungry. Can you tell me which pizzas are available?"*

How can we handle this?

## Classifying multiple intents

One way that we can handle this is by splitting the user request into multiple sentences. 

Given the sentence *"I would like to order two medium pizza since we are quite hungry. Can you tell me which pizzas are availale?"*

We could separate this into
1. *"I would like to order two medium pizza since we are quite hungry."*
2. *"Can you tell me which pizzas are availale?"*

Then, we can answer the user's request.

*Question*: How do we split the sentences?

In [2]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.94it/s]


In [10]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

task_prompt = """You are given a user input in natural language.
Break the user input into multiple sentences, each representing a single intent:
- pizza_ordering, if the user wants to order a pizza.
- drink_ordering, if the user wants to order a drink.
- request_info, if the user wants to know which pizzas or drinks (type, size) are available.
- out_of_domain, if the input does not match any of the above.

Provide a list of sentences, each enclosed in double quotes and separated by commas.
For example, given the input: "I want to order a large pepperoni pizza and also tell me what drinks you have."
You should output:
["I want to order a large pepperoni pizza", "Tell me what drinks you have"]

Only provide the list of sentences (not the intents) as output, without any additional text.
"""

In [11]:
user_message = "I would like to order two medium pizza since we are quite hungry. Can you tell me which pizzas are available?"

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(user_message, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs, max_new_tokens=16384).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, user_message, content)


### Conversation

**System:** You are given a user input in natural language.
Break the user input into multiple sentences, each representing a single intent:
- pizza_ordering, if the user wants to order a pizza.
- drink_ordering, if the user wants to order a drink.
- request_info, if the user wants to know which pizzas or drinks (type, size) are available.
- out_of_domain, if the input does not match any of the above.

Provide a list of sentences, each enclosed in double quotes and separated by commas.
For example, given the input: "I want to order a large pepperoni pizza and also tell me what drinks you have."
You should output:
["I want to order a large pepperoni pizza", "Tell me what drinks you have"]

Only provide the list of sentences (not the intents) as output, without any additional text.


**User:** I would like to order two medium pizza since we are quite hungry. Can you tell me which pizzas are available?

**Assistant:** ["I would like to order two medium pizza", "Tell me which pizzas are available"]

In [ ]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.94it/s]


In [12]:
user_message = "I would like to order two medium pizza and two drinks"

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(user_message, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs, max_new_tokens=16384).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, user_message, content)


### Conversation

**System:** You are given a user input in natural language.
Break the user input into multiple sentences, each representing a single intent:
- pizza_ordering, if the user wants to order a pizza.
- drink_ordering, if the user wants to order a drink.
- request_info, if the user wants to know which pizzas or drinks (type, size) are available.
- out_of_domain, if the input does not match any of the above.

Provide a list of sentences, each enclosed in double quotes and separated by commas.
For example, given the input: "I want to order a large pepperoni pizza and also tell me what drinks you have."
You should output:
["I want to order a large pepperoni pizza", "Tell me what drinks you have"]

Only provide the list of sentences (not the intents) as output, without any additional text.


**User:** I would like to order two medium pizza and two drinks

**Assistant:** ["I would like to order two medium pizza", "I would like to order two drinks"]

### Question

Do you see any issues with the following method?

Let's see an example: *"Hello, I would like to order a margherita pizza and a large pepperoni. Also, I would like to order two XL drinks. Can you tell me which ones are available? Please make this takeaway.*

In [14]:
user_message = "Hello, I would like to order a margherita pizza and a large pepperoni. Also, I would like to order two XL drinks. Can you tell me which ones are available? Please make this takeaway."

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(user_message, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs, max_new_tokens=16384).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, user_message, content)


### Conversation

**System:** You are given a user input in natural language.
Break the user input into multiple sentences, each representing a single intent:
- pizza_ordering, if the user wants to order a pizza.
- drink_ordering, if the user wants to order a drink.
- request_info, if the user wants to know which pizzas or drinks (type, size) are available.
- out_of_domain, if the input does not match any of the above.

Provide a list of sentences, each enclosed in double quotes and separated by commas.
For example, given the input: "I want to order a large pepperoni pizza and also tell me what drinks you have."
You should output:
["I want to order a large pepperoni pizza", "Tell me what drinks you have"]

Only provide the list of sentences (not the intents) as output, without any additional text.


**User:** Hello, I would like to order a margherita pizza and a large pepperoni. Also, I would like to order two XL drinks. Can you tell me which ones are available? Please make this takeaway.

**Assistant:** ["I would like to order a margherita pizza and a large pepperoni", "I would like to order two XL drinks", "Can you tell me which ones are available?", "Please make this takeaway"]

<details>
    <summary>The system starts to answer, processes the requests one by one, crafts the answers and...</summary><br>
    the user gets overwhelmed. Now our system has become over-informative, and we do not want this!<br><br>
    How do we solve this?
</details>

#### Solution

To solve this, we have to decide what to answer, and in which order.

<details>
    <summary>Ideas?</summary>
    <ul>
        <li>We could decide to answer only to a few or summarize the full answer</li>
        <li>We could answer the user request from the last part of the input (most recent).</li>
    <ul>
</details>

## Mixed-Initiative

Mixed initiative dialogues are usually more engaining and require less turns to fulfill the user's intent.

How can we introduce them in our system?

In [15]:
task_prompt = """You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
"""

In [16]:
mi = """Instead of simply answering the user's request or requesting more information, try to proactively suggest some options.
Pick from the following options:
- pizza_type: Margherita, Pepperoni, BBQ Chicken, Veggie, Hawaiian, Four Cheese
- pizza_size: Small, Medium, Large, Extra Large
- drink_type: Cola, Lemonade, Iced Tea
- drink_size: Small, Medium, Large, Extra Large
"""

In [17]:
nlg_input = """NBA: request_info(pizza_size)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt + mi
    }
]

text = prepare_text(nlg_input, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    # max_new_tokens limits the length of the generated response
    generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, nlg_input, content)


### Conversation

**System:** You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
Instead of simply answering the user's request or requesting more information, try to proactively suggest some options.
Pick from the following options:
- pizza_type: Margherita, Pepperoni, BBQ Chicken, Veggie, Hawaiian, Four Cheese
- pizza_size: Small, Medium, Large, Extra Large
- drink_type: Cola, Lemonade, Iced Tea
- drink_size: Small, Medium, Large, Extra Large


**User:** NBA: request_info(pizza_size)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}

**Assistant:** How about a Medium or Large Margherita pizza? Medium is great for sharing, while Large is perfect for a hearty meal. Would you like to go with one of these sizes?

In [19]:
nlg_input = """NBA: request_info(drink_type)
DS: {
  "intent": "drink_ordering",
  "slots": {
    "drink_size": null,
    "drink_type": null,
    "drink_count": null
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt + mi
    }
]

text = prepare_text(nlg_input, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    # max_new_tokens limits the length of the generated response
    generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, nlg_input, content)


### Conversation

**System:** You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
Instead of simply answering the user's request or requesting more information, try to proactively suggest some options.
Pick from the following options:
- pizza_type: Margherita, Pepperoni, BBQ Chicken, Veggie, Hawaiian, Four Cheese
- pizza_size: Small, Medium, Large, Extra Large
- drink_type: Cola, Lemonade, Iced Tea
- drink_size: Small, Medium, Large, Extra Large


**User:** NBA: request_info(drink_type)
DS: {
  "intent": "drink_ordering",
  "slots": {
    "drink_size": null,
    "drink_type": null,
    "drink_count": null
  }
}

**Assistant:** Would you like a Cola, Lemonade, or Iced Tea? We also offer Small, Medium, Large, or Extra Large sizes. Want to try our popular Lemonade with a twist? 😊

## Confirmation, Grounding, and Conversational Markers

Providing valuable feedback to the user is quite important, so we should take care of that and ensure that our system:
- ask the user for a confirmation (e.g., *"Is that correct?"* or *"Should I place the order?"*)
- acknowledge what the user said (e.g., *"So you would like one margherita pizza"*)
- uses conversational markers (e.g., *"Great"*, *"Perfect!"*, or *"Last step"*)

So, let's modify the prompt in order to do this!

In [ ]:
# Write your code below

## What about the history

So far we have only passed information such as the Dialogue State (DS) or the Next Best Action (NBA).

However, sometimes additional information could improve the interaction.

For example, if the user says *"I said I wanted a large pizza, not medium"*
1. the NLU will collect the information about the new value for the slot
2. the DM will (possibly) confirm the order
3. and the NLG will answer something like *"Got it, so you want a large pizza"*

However, it would be better to say *"I apologize for the confusion, I've changed the order to a large pizza"*

We can do this using the dialogue history, which contains the previous exchanges between the user and the system.

In this case, only including the last turn could be sufficient. Of course, the context window depends on the application, and the system capabilities may deteriorate if the context becomes too long.

In [ ]:
# Try to implement this by passing the last user message to the system prompt